# Serverless LLMs and Agentic AI with Modal – Lesson 7
## Web Endpoints — Exposing Your LLM as a Public HTTP API

In Lesson 6 you built a `TextGenerator` Modal Class that loads `distilgpt2` once per container and serves many `.generate()` calls. That works great from Python — but how does the rest of the world (a frontend, a mobile app, an agent in another runtime) actually *call* your model?

The answer is **web endpoints**. Modal can take any function or class method and expose it at a public HTTPS URL with zero extra infrastructure.

In this lesson you'll learn the two main patterns:

1. **`@modal.fastapi_endpoint`** — a one-line decorator that turns a function (or class method) into a single HTTP endpoint. Perfect for quick, focused APIs.
2. **`@modal.asgi_app`** — wrap a full **FastAPI** app to get multiple endpoints, request validation with Pydantic, automatic OpenAPI docs, and middleware.

### Colab-friendly note
Modal has two ways to run a web app: `modal serve` (dev URL with hot reload, **blocks the terminal**) and `modal deploy` (persistent URL, **exits after deploying**). Because Colab runs one cell at a time, we'll use **`modal deploy`** throughout — that way every cell can finish and the next cell can hit the endpoint with `requests`. If you later work locally, `modal serve` is the better dev loop.

### What you'll create
A script `lesson7_web_endpoints_llm.py` that contains:

1. The `TextGenerator` class from Lesson 6, **with a `@modal.fastapi_endpoint` method** on it (Pattern A).
2. A full FastAPI app exposed via `@modal.asgi_app()` with `/generate`, `/health`, and `/batch` routes plus Pydantic request models (Pattern B).
3. End-to-end testing from the same notebook: deploy → discover URLs → hit them with `requests` → clean up.


In [ ]:
# =====================================
# Step 0 – Install Modal + requests
# =====================================
!pip install modal requests --quiet
!which modal
!modal --version
print("✅ Modal installed.")

## Step 1 – Verify authentication


In [ ]:
# ============================
# Step 1 – Configure Modal using the CLI (matches docs)
# ============================
# ⚠️ IMPORTANT:
# - Replace the placeholder strings with your real MODAL_TOKEN_ID and MODAL_TOKEN_SECRET.
# - Do NOT commit these values to GitHub or share them.

TOKEN_ID = ""        # <-- paste from Modal dashboard
TOKEN_SECRET = ""    # <-- paste from Modal dashboard

if not TOKEN_ID or not TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

print("✅ Token stored via `modal token set`. You should be authenticated now.")

## Step 2 – Write the Lesson 7 app

We keep the same model from Lesson 6 (`distilgpt2`) but now we attach **HTTP endpoints** to it.

### Two patterns side-by-side

**Pattern A — single endpoint on a class method (simple):**
```python
@app.cls(gpu="T4")
class TextGenerator:
    @modal.enter()
    def load(self): ...

    @modal.fastapi_endpoint(method="POST", docs=True)
    def web_generate(self, prompt: str = "Hello"): ...
```
The container that holds the GPU + model **also serves HTTP directly**. One hop, fewer moving parts.

**Pattern B — full FastAPI app routed to the class (production):**
```python
@app.function()
@modal.asgi_app()
def api():
    web_app = FastAPI(...)
    gen = TextGenerator()
    @web_app.post("/generate")
    def post_generate(req): return gen.generate.remote(...)
    return web_app
```
The API container is cheap (CPU only). It calls the GPU class **remotely**. Lets you scale the API tier and the inference tier independently — common production pattern.

We set up both in the same script so you can compare them.


In [ ]:
%%writefile lesson7_web_endpoints_llm.py
import time
from typing import Dict, List

import modal

# ------------------------------------------------------------
# Image: PyTorch + transformers + FastAPI, with model baked in
# ------------------------------------------------------------
MODEL_NAME = "distilgpt2"


def _download_model():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    AutoTokenizer.from_pretrained(MODEL_NAME)
    AutoModelForCausalLM.from_pretrained(MODEL_NAME)


image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "torch==2.3.0",
        "transformers==4.44.2",
        "fastapi[standard]==0.115.0",
        "pydantic==2.9.2",
    )
    .run_function(_download_model)
)

app = modal.App("lesson7-web-endpoints-llm", image=image)


# ============================================================
# PATTERN A — class method exposed as a single HTTP endpoint
# ============================================================
@app.cls(gpu="T4", scaledown_window=120)
class TextGenerator:
    """Same lifecycle as Lesson 6, but `web_generate` is ALSO an HTTPS endpoint."""

    @modal.enter()
    def load(self):
        print(f"[enter] loading {MODEL_NAME} ...")
        t0 = time.time()
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(self.device)
        self.model.eval()
        print(f"[enter] model ready on {self.device} in {time.time()-t0:.2f}s")

    # ---- Internal method, callable from Python and from the FastAPI app ----
    @modal.method()
    def generate(self, prompt: str, max_new_tokens: int = 32) -> Dict:
        import torch
        t0 = time.time()
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        return {
            "prompt": prompt,
            "completion": text,
            "device": self.device,
            "latency_s": round(time.time() - t0, 3),
        }

    # ---- PATTERN A: single HTTP endpoint, served by THIS GPU container ----
    @modal.fastapi_endpoint(method="POST", docs=True)
    def web_generate(self, prompt: str = "Hello", max_new_tokens: int = 32) -> Dict:
        """POST ?prompt=...&max_new_tokens=...  ->  same JSON as .generate(), but over HTTPS."""
        return self.generate.local(prompt=prompt, max_new_tokens=max_new_tokens)


# ============================================================
# PATTERN B — full FastAPI app via @modal.asgi_app()
# ============================================================
# Lives on its own cheap CPU container, calls the GPU class remotely.
# Lets you scale the API tier independently from the GPU tier.
# ============================================================
@app.function()
@modal.asgi_app()
def api():
    from fastapi import FastAPI
    from pydantic import BaseModel, Field

    web_app = FastAPI(
        title="Lesson 7 – Serverless LLM API",
        version="1.0",
    )
    gen = TextGenerator()  # reference to the class; calls go to the GPU container

    class GenRequest(BaseModel):
        prompt: str = Field(..., description="Input prompt")
        max_new_tokens: int = Field(32, ge=1, le=256)

    class BatchRequest(BaseModel):
        prompts: List[str]
        max_new_tokens: int = Field(32, ge=1, le=256)

    @web_app.get("/health")
    def health():
        return {"status": "ok", "app": "lesson7-web-endpoints-llm"}

    @web_app.post("/generate")
    def post_generate(req: GenRequest):
        return gen.generate.remote(req.prompt, req.max_new_tokens)

    @web_app.post("/batch")
    def post_batch(req: BatchRequest):
        args = [(p, req.max_new_tokens) for p in req.prompts]
        results = list(gen.generate.starmap(args))
        return {"count": len(results), "results": results}

    return web_app


@app.local_entrypoint()
def lesson7_main():
    """Quick Python-side sanity check before we go HTTP."""
    print("\n========================================")
    print("Lesson 7 – Web Endpoints for your LLM")
    print("========================================\n")

    gen = TextGenerator()
    r = gen.generate.remote("A serverless LLM API is great because")
    print(r)
    print("\n✅ Next: deploy and call the endpoints from your notebook.")


## Step 3 – Sanity check the Python path first

Before touching HTTP, confirm the class itself still works (this is just Lesson 6). `modal run` builds the image the first time, then runs the entrypoint and **exits**.


In [ ]:
!modal run lesson7_web_endpoints_llm.py

## Step 4 – Deploy and capture the public URLs

`modal deploy` publishes your endpoints to **persistent** HTTPS URLs and then exits.
We tee the output to a log file so the next cell can parse the URLs automatically — no copy-paste needed.


In [ ]:
# Deploy and save the full output to /tmp/deploy.log
!modal deploy lesson7_web_endpoints_llm.py 2>&1 | tee /tmp/deploy.log

In [ ]:
# Parse the deploy log to find the two public URLs.
import re

with open("/tmp/deploy.log") as f:
    log = f.read()

urls = re.findall(r"https://[\w.-]+\.modal\.run", log)
urls = list(dict.fromkeys(urls))  # dedupe, preserve order
print("All endpoint URLs discovered:")
for u in urls:
    print("  ", u)

PATTERN_A_URL = next((u for u in urls if "web-generate" in u or "web_generate" in u), None)
API_URL       = next((u for u in urls if u.endswith("-api.modal.run") or u.endswith("--api.modal.run")), None)

# Fallback ordering if the names don't match (Modal occasionally tweaks slugs)
if PATTERN_A_URL is None and len(urls) >= 1:
    PATTERN_A_URL = urls[0]
if API_URL is None and len(urls) >= 2:
    API_URL = urls[1]

print("\n🔗 Pattern A (class method endpoint):", PATTERN_A_URL)
print("🔗 Pattern B (FastAPI app base URL): ", API_URL)

assert PATTERN_A_URL and API_URL, "❌ Could not parse both URLs from the deploy log — check the cell above."

## Step 5 – Call the endpoints from this notebook

Now `PATTERN_A_URL` and `API_URL` are real public HTTPS URLs.
We hit them with `requests` exactly like any third-party REST API.


In [ ]:
# --- Pattern B: health check ---
import requests, json, time

r = requests.get(f"{API_URL}/health", timeout=30)
print("GET /health ->", r.status_code, r.json())

In [ ]:
# --- Pattern B: single generation ---
# First call is COLD (container boot + model load).
t0 = time.time()
r = requests.post(
    f"{API_URL}/generate",
    json={"prompt": "Serverless LLM APIs are useful because", "max_new_tokens": 32},
    timeout=180,
)
print(f"POST /generate ({time.time()-t0:.2f}s, HTTP {r.status_code}):")
print(json.dumps(r.json(), indent=2))

In [10]:
# --- Pattern B: batch generation (fans out via .starmap inside the API) ---
t0 = time.time()
r = requests.post(
    f"{API_URL}/batch",
    json={
        "prompts": [
            "Tool use means",
            "Retrieval augmented generation is",
            "An agent loop typically",
        ],
        "max_new_tokens": 24,
    },
    timeout=180,
)
print(f"POST /batch ({time.time()-t0:.2f}s, HTTP {r.status_code}):")
print(json.dumps(r.json(), indent=2))

POST /batch (1.39s, HTTP 200):
{
  "count": 3,
  "results": [
    {
      "prompt": "Tool use means",
      "completion": "Tool use means to use the same method.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n",
      "device": "cuda",
      "latency_s": 0.102
    },
    {
      "prompt": "Retrieval augmented generation is",
      "completion": "Retrieval augmented generation is a new technology that is being developed by the University of California, Berkeley.\n\n\n\n\n\n\n\n\n",
      "device": "cuda",
      "latency_s": 0.103
    },
    {
      "prompt": "An agent loop typically",
      "completion": "An agent loop typically takes a few seconds to complete.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n",
      "device": "cuda",
      "latency_s": 0.102
    }
  ]
}


In [12]:
# --- Pattern A: hit the class-method endpoint directly ---
# This one takes prompt + max_new_tokens as QUERY params (that's how @modal.fastapi_endpoint
# maps simple typed args by default).
t0 = time.time()
r = requests.post(
    PATTERN_A_URL,
    params={"prompt": "Pattern A endpoints are simplest when", "max_new_tokens": 32},
    timeout=180,
)
print(f"POST {PATTERN_A_URL} ({time.time()-t0:.2f}s, HTTP {r.status_code}):")
print(json.dumps(r.json(), indent=2))

POST https://farhad-rh--lesson7-web-endpoints-llm-api.modal.run (3.71s, HTTP 404):
{
  "detail": "Not Found"
}


In [13]:
print("Interactive docs:", API_URL + "/docs")

Interactive docs: https://farhad-rh--lesson7-web-endpoints-llm-api.modal.run/docs


## Step 6 – Cleanup

A deployed app keeps its URL alive and can incur idle costs (especially with `min_containers > 0`). When you're done experimenting, stop it.


In [ ]:
!modal app list

In [ ]:
!modal app stop lesson7-web-endpoints-llm